# 🗺️ Knowledge Graph & Path Generation (OULAD Dataset)
---
Notebook ini berfokus pada **Fase 2** dari roadmap AI Learning Path. Di sini kita akan:
1. Membangun **Knowledge Graph** (Peta Konsep) dari struktur modul dan aktivitas di OULAD.
2. Mengekstrak **Student State** berdasarkan riwayat nyata (log VLE dan asesmen).
3. Membuat algoritma **Path Generation** menggunakan _Topological Sort_ untuk merekomendasikan aktivitas selanjutnya yang paling relevan bagi mahasiswa.

### 1. Setup & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import os
from collections import defaultdict

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

# Menghubungkan ke Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/AI-Learning Path/'
except:
    print("Tidak berjalan di Google Colab. Menggunakan path lokal.")
    BASE_PATH = './'

print("✅ Setup Selesai!")

### 2. Memuat Data OULAD & Hasil Agregasi

In [ ]:
%%time

# Memuat dataset asli OULAD
courses = pd.read_csv(os.path.join(BASE_PATH, 'courses.csv'))
assessments = pd.read_csv(os.path.join(BASE_PATH, 'assessments.csv'))
vle = pd.read_csv(os.path.join(BASE_PATH, 'vle.csv'))
studentInfo = pd.read_csv(os.path.join(BASE_PATH, 'studentInfo.csv'))
studentAssessment = pd.read_csv(os.path.join(BASE_PATH, 'studentAssessment.csv'))

# Untuk studentVle, kita muat subset jika terlalu besar, atau agregat
# Disini kita ambil subset kecil untuk eksperimen (misalnya 1 juta baris pertama)
print("Memuat studentVle.csv...")
studentVle = pd.read_csv(os.path.join(BASE_PATH, 'studentVle.csv'), nrows=1000000)

print(f"Data berhasil dimuat. studentVle (sampel): {studentVle.shape}")

### 3. Membangun KnowledgeGraphBuilder
Kelas ini akan mengubah struktur kursus dan aktivitas VLE menjadi _Directed Acyclic Graph_ (DAG).

In [ ]:
class KnowledgeGraphBuilder:
    """Membangun Knowledge Graph dari struktur VLE OULAD"""
    
    def __init__(self):
        self.G = nx.DiGraph()
        
    def build_from_oulad(self, courses, assessments, vle, student_vle, target_module='AAA'):
        """
        Membangun graph aktivitas untuk satu modul spesifik.
        - Nodes: id_site (aktivitas VLE) dan id_assessment (Tugas/Ujian)
        - Edges: Urutan waktu pengerjaan mayoritas mahasiswa
        """
        print(f"Membangun graph untuk Modul {target_module}...")
        
        # Filter data untuk modul tersebut
        vle_mod = vle[vle['code_module'] == target_module].copy()
        asst_mod = assessments[assessments['code_module'] == target_module].copy()
        svle_mod = student_vle[student_vle['code_module'] == target_module].copy()
        
        # 1. Tambahkan nodes untuk aktivitas VLE
        for _, row in vle_mod.iterrows():
            node_id = f"VLE_{row['id_site']}"
            self.G.add_node(node_id, type=row['activity_type'], 
                            week_from=row['week_from'], difficulty=0.5)
            
        # 2. Tambahkan nodes untuk Assessments
        for _, row in asst_mod.iterrows():
            node_id = f"ASSESS_{row['id_assessment']}"
            self.G.add_node(node_id, type=row['assessment_type'], 
                            weight=row['weight'], difficulty=0.8)
            
        # 3. Inferensi Edge (Prasyarat) berdasarkan waktu (date)
        # Agregat waktu rata-rata siswa mengakses materi
        avg_time_vle = svle_mod.groupby('id_site')['date'].mean().sort_values()
        
        prev_node = None
        for id_site, avg_date in avg_time_vle.items():
            curr_node = f"VLE_{id_site}"
            if curr_node in self.G.nodes():
                if prev_node:
                    self.G.add_edge(prev_node, curr_node)
                prev_node = curr_node
                
        print(f"Knowledge Graph selesai: {self.G.number_of_nodes()} Nodes, {self.G.number_of_edges()} Edges")
        return self.G
    
    def validate(self):
        """Memeriksa apakah graph merupakan DAG yang valid"""
        is_dag = nx.is_directed_acyclic_graph(self.G)
        print(f"Apakah graph ini DAG valid? {'✅ Ya' if is_dag else '❌ Tidak'}")
        return is_dag
    
    def get_learning_order(self):
        """Mendapatkan urutan pembelajaran (Topological Sort)"""
        try:
            return list(nx.topological_sort(self.G))
        except nx.NetworkXUnfeasible:
            print("Graph memiliki cycle, tidak bisa melakukan topological sort.")
            return []
            
    def visualize(self, max_nodes=50):
        """Visualisasi Graph (dibatasi agar tidak terlalu padat)"""
        plt.figure(figsize=(15, 10))
        
        if self.G.number_of_nodes() > max_nodes:
            nodes_to_draw = list(self.G.nodes())[:max_nodes]
            sub_G = self.G.subgraph(nodes_to_draw)
        else:
            sub_G = self.G
            
        pos = nx.spring_layout(sub_G, seed=42)
        nx.draw(sub_G, pos, with_labels=False, node_color='skyblue', 
                node_size=150, arrowsize=10, edge_color='gray', alpha=0.6)
        plt.title(f"Knowledge Graph Visualisasi (Sample {sub_G.number_of_nodes()} Nodes)")
        plt.show()

kg_builder = KnowledgeGraphBuilder()
G = kg_builder.build_from_oulad(courses, assessments, vle, studentVle, target_module='AAA')
kg_builder.validate()
kg_builder.visualize()

### 4. Student State (Melacak Penguasaan Mahasiswa)
Kita membangun class untuk melacak interaksi mahasiswa spesifik.

In [ ]:
class StudentState:
    def __init__(self, student_id, knowledge_graph):
        self.student_id = student_id
        self.kg = knowledge_graph
        self.mastery = {node: 0.0 for node in self.kg.nodes()} # Default 0%
        
    def load_from_oulad(self, student_vle, student_assessment):
        """Memuat log VLE dan assessment untuk student ini"""
        svle = student_vle[student_vle['id_student'] == self.student_id]
        sass = student_assessment[student_assessment['id_student'] == self.student_id]
        
        # Asumsi: Jika pernah klik materi, kita beri skor mastery awal (misal 0.5)
        for _, row in svle.iterrows():
            node_id = f"VLE_{row['id_site']}"
            if node_id in self.mastery:
                # Semakin banyak klik, semakin besar perkiraan penguasaan
                click_score = min(1.0, row['sum_click'] / 10.0) 
                self.mastery[node_id] = max(self.mastery[node_id], click_score)
                
        # Asumsi: Untuk assessment, mastery = score / 100
        for _, row in sass.iterrows():
            node_id = f"ASSESS_{row['id_assessment']}"
            if node_id in self.mastery:
                self.mastery[node_id] = row['score'] / 100.0
                
        print(f"Data dimuat untuk siswa {self.student_id}. Aktivitas tercatat: {len(svle)} VLE, {len(sass)} Asesmen.")
        
    def get_unmastered(self, threshold=0.8):
        return [node for node, score in self.mastery.items() if score < threshold]
        
# Uji coba untuk satu mahasiswa
sample_student = studentInfo[studentInfo['code_module'] == 'AAA']['id_student'].iloc[0]
state = StudentState(sample_student, G)
state.load_from_oulad(studentVle, studentAssessment)

### 5. Personalized Path Generation
Algoritma AI untuk menyusun rekomendasi pembelajaran selanjutnya (adaptive path).

In [ ]:
class PersonalizedPathGenerator:
    def __init__(self, knowledge_graph):
        self.G = knowledge_graph
        
    def generate_path(self, student_state, threshold=0.8, n_recommendations=5):
        """
        Mencari aktivitas yang belum dikuasai (mastery < threshold), 
        namun prasyaratnya sudah terpenuhi.
        """
        unmastered = student_state.get_unmastered(threshold)
        
        ready_concepts = []
        for node in unmastered:
            preds = list(self.G.predecessors(node))
            # Cek apakah semua predecessor sudah dikuasai
            if all(student_state.mastery.get(p, 0.0) >= threshold for p in preds):
                ready_concepts.append({
                    'node_id': node,
                    'type': self.G.nodes[node].get('type', 'Unknown'),
                    'current_mastery': student_state.mastery.get(node, 0.0)
                })
                
        # Jika DAG memiliki urutan (Topological Sort)
        try:
            learning_order = list(nx.topological_sort(self.G))
            # Urutkan ready_concepts berdasarkan learning_order
            ready_concepts.sort(key=lambda x: learning_order.index(x['node_id']) if x['node_id'] in learning_order else 9999)
        except nx.NetworkXUnfeasible:
            pass # Jika ada cycle, abaikan topological sort
            
        return ready_concepts[:n_recommendations]

path_gen = PersonalizedPathGenerator(G)
recs = path_gen.generate_path(state)

print("\n\U0001f3af Rekomendasi Belajar Selanjutnya:")
for i, rec in enumerate(recs, 1):
    print(f"{i}. {rec['node_id']} (Type: {rec['type']}) - Mastery Saat Ini: {rec['current_mastery']:.0%}")

### 6. Kesimpulan & Insight
Model grafik berhasil mengambil urutan VLE nyata dan merangkai DAG (Peta Konsep). Dengan ini, sistem AI bisa mengenali di mana posisi mahasiswa, dan modul apa yang seharusnya diajarkan berikutnya berdasarkan prasyarat yang ada.